# 04. Carga a staging en Parquet — `sa_dataset_obras_publicas`

Lee los snapshots mensuales `DataSet-Obras-Publicas DD-MM-YYYY.xlsx` de `data/raw/cgr`, selecciona el mapping según la fecha del nombre: el original para fotos **anteriores al 01-07-2026** y `stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml` para fotos **desde el 01-07-2026 inclusive** y escribe **una partición diaria por archivo** con **layout Hive** (`fecha_foto=YYYY-MM-DD`) en `data/stg/sa_dataset_obras_publicas/`.

```
data/stg/
├── sa_dataset_obras_publicas/            <- dataset limpio: solo particiones
│   ├── fecha_foto=2026-01-01/part-00000.parquet
│   ├── fecha_foto=2026-02-01/part-00000.parquet
│   └── ...
├── _work/sa_dataset_obras_publicas/      <- temporales / rechazados (fuera del dataset)
└── logs/                                 <- log de ejecución (.log) + resumen (.csv) por corrida
```

**Convención Hive**: `fecha_foto` es la **columna de partición**: vive en el nombre de la carpeta y **no se repite dentro del Parquet**. Al leer el dataset completo con `hive_partitioning = true` reaparece como columna `DATE` y permite filtrar leyendo solo las carpetas necesarias:

```sql
SELECT * FROM read_parquet('data/stg/sa_dataset_obras_publicas/*/*.parquet', hive_partitioning = true)
WHERE fecha_foto BETWEEN '2025-06-01' AND '2025-12-01'
```

**Modos de ejecución** (parámetro `MODO`)
- `PRUEBA`: procesa **un solo archivo** (`FECHA_PRUEBA`), ejecuta todas las validaciones y muestra el detalle. Úselo antes de la carga masiva.
- `COMPLETA`: procesa todos los archivos del rango `FECHA_INICIO`–`FECHA_FIN` (o todos si son `None`).

**Reglas de carga** (del mapping, `load.strategy = snapshot_replace`)
- **Idempotente por partición**: con `SOBRESCRIBIR = True` se reemplaza `p_YYYYMMDD` si ya existía; con `False` se omite.
- **Escritura segura**: se escribe en `data/stg/_work/.../_tmp`, se valida, y solo si no hay validaciones en `FALLA` se promueve a `fecha_foto=YYYY-MM-DD` (la partición anterior se conserva si algo falla; lo rechazado queda en `_work/.../_error` para inspección). Los temporales **nunca** están dentro de la carpeta del dataset.
- **Esquema explícito** tomado del mapping (`string→VARCHAR`, `integer→INTEGER`, `decimal→DOUBLE`, `boolean→BOOLEAN`, `date→DATE`), igual en todas las particiones. El archivo Parquet contiene todas las columnas **excepto** la de partición.
- Columnas de auditoría según `active` en `audit_columns` (`fecha_foto` activa ⇒ columna de partición).
- **Compresión**: ZSTD (nivel configurable) y filas ordenadas por `codigo_infobras` dentro de cada archivo (mejor compresión y *row-group skipping* al buscar una obra).
- Se usa **DuckDB** para escribir el Parquet (no requiere `pyarrow`).

**Trazabilidad**: cada corrida genera un `.log` con parámetros, versiones, y por archivo: ruta, tamaño, **SHA-256**, fila de cabecera, *fecha de consulta* declarada en el Excel, filas leídas/cargadas/descartadas, fallas de conversión, resultado de cada validación y tiempos.

## 1. Parámetros

In [ ]:
# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================
MODO = "PRUEBA"            # "PRUEBA" (un archivo) | "COMPLETA" (rango de fechas)
FECHA_PRUEBA = "20260711"  # YYYYMMDD: archivo a procesar en modo PRUEBA

FECHA_INICIO = "20230711"        # YYYYMMDD o None (= desde el primer archivo)   [solo MODO = "COMPLETA"]
FECHA_FIN = "20230711"           # YYYYMMDD o None (= hasta el último archivo)   [solo MODO = "COMPLETA"]

SOBRESCRIBIR = True        # True: reemplaza la partición si ya existe | False: la omite
DETENER_ANTE_ERROR = True  # True: detiene la corrida ante el primer archivo con ERROR/RECHAZADO
FILA_CABECERA = None       # None = autodetectar (fila con "Código INFOBRAS") | entero (1-based)

# Formato Parquet
COLUMNA_PARTICION = "fecha_foto"                                  # columna de partición Hive (no se escribe dentro del archivo)
ORDENAR_POR = ["codigo_infobras", "ano_avance", "mes_avance"]     # orden de filas dentro del archivo (mejora compresión y filtros)
COMPRESION = "ZSTD"
COMPRESION_NIVEL = 9                                              # ZSTD 1-22 (mayor = más compacto, escritura más lenta)

# Rutas (None = valores por defecto del proyecto)
DIR_ORIGEN = None          # por defecto: data/raw/cgr
DIR_DESTINO = None         # por defecto: data/stg/sa_dataset_obras_publicas
DIR_LOGS = None            # por defecto: data/stg/logs
# ==============================================================================

## 2. Configuración, log de ejecución y mapping

In [2]:
import getpass
import gc
import hashlib
import logging
import platform
import re
import shutil
import socket
import sys
import time
import traceback
from datetime import date, datetime
from pathlib import Path

import duckdb
import openpyxl
import pandas as pd
import yaml

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)

MODO = str(MODO).upper()
assert MODO in ("PRUEBA", "COMPLETA"), 'MODO debe ser "PRUEBA" o "COMPLETA"'


def obtener_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "docs" / "datamappings").exists():
            return carpeta
    return actual


PROJECT_ROOT = obtener_raiz_proyecto()
STG_DIR = PROJECT_ROOT / "data" / "stg"
RAW_DIR = Path(DIR_ORIGEN) if DIR_ORIGEN else PROJECT_ROOT / "data" / "raw" / "cgr"
DEST_DIR = Path(DIR_DESTINO) if DIR_DESTINO else STG_DIR / "sa_dataset_obras_publicas"
WORK_DIR = DEST_DIR.parent / "_work" / DEST_DIR.name   # temporales/rechazados: fuera del dataset
LOGS_DIR = Path(DIR_LOGS) if DIR_LOGS else STG_DIR / "logs"
MAPPING_PATH_ANTERIOR = PROJECT_ROOT / "docs" / "datamappings" / "stg__dataset_obras_publicas__to__sa_obras_publicas_infobras.yaml"

MAPPING_PATH_NUEVO = PROJECT_ROOT / "docs" / "datamappings" / "stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml"
with open(MAPPING_PATH_ANTERIOR, encoding="utf-8") as f:
    MAPPING_ANTERIOR = yaml.safe_load(f)
with open(MAPPING_PATH_NUEVO, encoding="utf-8") as f:
    MAPPING_NUEVO = yaml.safe_load(f)
FECHA_CAMBIO_MAPPING = date.fromisoformat(MAPPING_NUEVO["source"]["validity"]["snapshot_date_from_inclusive"])


def seleccionar_mapping(fecha_foto):
    """Selecciona por la fecha del nombre del archivo, con corte inclusivo."""
    if pd.Timestamp(fecha_foto).date() >= FECHA_CAMBIO_MAPPING:
        return MAPPING_NUEVO, MAPPING_PATH_NUEVO
    return MAPPING_ANTERIOR, MAPPING_PATH_ANTERIOR


DEST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = LOGS_DIR / f"carga_stg_sa_dataset_obras_publicas_{MODO.lower()}_{RUN_ID}.log"
RESUMEN_PATH = LOGS_DIR / f"carga_stg_sa_dataset_obras_publicas_{MODO.lower()}_{RUN_ID}_resumen.csv"


def configurar_logger() -> logging.Logger:
    """Logger con salida a consola y a archivo (se reinicia en cada ejecución del notebook)."""
    logger = logging.getLogger("carga_stg")
    for h in list(logger.handlers):
        h.close()
        logger.removeHandler(h)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
    sh = logging.StreamHandler(sys.stdout)
    for h in (fh, sh):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


log = configurar_logger()


log.info("=" * 100)
log.info("INICIO DE CORRIDA | run_id=%s | modo=%s", RUN_ID, MODO)
log.info("Usuario=%s | Host=%s | SO=%s", getpass.getuser(), socket.gethostname(), platform.platform())
log.info("Python=%s | pandas=%s | duckdb=%s | openpyxl=%s", platform.python_version(), pd.__version__, duckdb.__version__, openpyxl.__version__)
for documento, ruta_mapping in ((MAPPING_ANTERIOR, MAPPING_PATH_ANTERIOR), (MAPPING_NUEVO, MAPPING_PATH_NUEVO)):
    log.info("Mapping disponible: %s v%s (status=%s) | %s", documento["mapping"]["id"], documento["mapping"]["version"], documento["mapping"]["status"], ruta_mapping.name)
log.info("Cambio de estructura: %s (inclusive)", FECHA_CAMBIO_MAPPING)
log.info("Origen : %s", RAW_DIR)
log.info("Destino: %s (layout Hive: %s=YYYY-MM-DD)", DEST_DIR, COLUMNA_PARTICION)
log.info("Trabajo: %s", WORK_DIR)
log.info("Log    : %s", LOG_PATH)
log.info("Parquet: compresión=%s nivel=%s | orden=%s", COMPRESION, COMPRESION_NIVEL, ORDENAR_POR)
ajenos = [d.name for d in DEST_DIR.iterdir() if not (d.is_dir() and d.name.startswith(f"{COLUMNA_PARTICION}="))]
if ajenos:
    log.warning("El dataset contiene elementos que NO son particiones Hive (afectan la lectura del dataset completo): %s", ajenos)
log.info("Parámetros: FECHA_PRUEBA=%s FECHA_INICIO=%s FECHA_FIN=%s SOBRESCRIBIR=%s DETENER_ANTE_ERROR=%s FILA_CABECERA=%s",
         FECHA_PRUEBA, FECHA_INICIO, FECHA_FIN, SOBRESCRIBIR, DETENER_ANTE_ERROR, FILA_CABECERA)

2026-11-17 10:16:45 [INFO] ====================================================================================================
2026-11-17 10:16:45 [INFO] INICIO DE CORRIDA | run_id=20261117_101645 | modo=COMPLETA
2026-11-17 10:16:45 [INFO] Usuario=yanfr | Host=LAPTOP-0QLBKUN0 | SO=Windows-10-10.0.19045-SP0
2026-11-17 10:16:45 [INFO] Python=3.13.7 | pandas=3.0.5 | duckdb=2.0.0 | openpyxl=3.1.5
2026-11-17 10:16:45 [INFO] Mapping disponible: map_cgr_infobras_obras_publicas v1.0.0 (status=draft) | stg__dataset_obras_publicas__to__sa_obras_publicas_infobras.yaml
2026-11-17 10:16:45 [INFO] Mapping disponible: map_cgr_infobras_obras_publicas_desde_20260701 v2.0.0 (status=draft) | stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml
2026-11-17 10:16:45 [INFO] Cambio de estructura: 2026-07-01 (inclusive)
2026-11-17 10:16:45 [INFO] Origen : C:\ws\formacion\uni-mia\mia-tesis-opa\data\raw\cgr
2026-11-17 10:16:45 [INFO] Destino: C:\ws\formacion\uni-mia\mia-tesis-opa\dat

## 3. Mapping de columnas y esquema destino

Cada archivo usa su versión del mapping. La estructura anterior resuelve encabezados por nombre y ocurrencia; la nueva usa `source_position` (1-based) y verifica el encabezado, admitiendo el sufijo `.1` que pandas añade a duplicados.

El esquema Parquet común conserva los destinos del mapping anterior e incorpora cualquier destino confirmado adicional. Los destinos sin fuente se completan con `NULL`. Las columnas con `target: null` quedan excluidas y se registran como pendientes; `proposed_target` no se usa como destino hasta su confirmación en el mapping.


In [3]:
def norm_header(texto) -> str:
    t = re.sub(r"\s+", " ", str(texto)).strip().lower()
    return t.rstrip(":").strip()


def preparar_columnas_mapping(documento):
    columnas = []
    for grupo in documento["column_groups"]:
        for col in grupo["columns"]:
            if col.get("target") is None:
                continue
            columnas.append({
                "grupo": grupo["group"], "source": col["source"],
                "source_norm": norm_header(col["source"]),
                "occurrence": col.get("source_occurrence", 1),
                "position": col.get("source_position"),
                "target": col["target"], "type": col["type"], "transform": col["transform"],
            })
    return pd.DataFrame(columnas)


MAPAS_COLUMNAS = {doc["mapping"]["id"]: preparar_columnas_mapping(doc)
                  for doc in (MAPPING_ANTERIOR, MAPPING_NUEVO)}
# La unión conserva orden y tipos históricos para leer todas las particiones juntas.
columnas_comunes = {}
for tabla in MAPAS_COLUMNAS.values():
    assert tabla["target"].is_unique, "Destinos duplicados dentro de un mapping"
    for r in tabla.itertuples():
        assert r.target not in columnas_comunes or columnas_comunes[r.target] == r.type, f"Tipo incompatible para {r.target}"
        columnas_comunes[r.target] = r.type

# Auditoría: solo las columnas con active: true
AUDIT_GENERATORS = {
    "fecha_foto": lambda ctx: ctx["fecha_foto"],
    "nombre_archivo": lambda ctx: ctx["nombre_archivo"],
    "fecha_creacion": lambda ctx: pd.Timestamp.now().floor("s"),
    "usuario_creacion": lambda ctx: getpass.getuser(),
}
AUDIT_ACTIVE = [a for a in MAPPING_ANTERIOR["audit_columns"] if a.get("active", True)]
AUDIT_IGNORED = [a["target"] for a in MAPPING_ANTERIOR["audit_columns"] if not a.get("active", True)]
assert not (set(a["target"] for a in AUDIT_ACTIVE) - set(AUDIT_GENERATORS)), "Columna de auditoría activa sin generador"

TIPO_DUCKDB = {"string": "VARCHAR", "integer": "INTEGER", "decimal": "DOUBLE", "boolean": "BOOLEAN",
               "date": "DATE", "timestamp": "TIMESTAMP"}

# Esquema destino ordenado: columnas del mapping + auditoría activa
SCHEMA = [(c, TIPO_DUCKDB[t]) for c, t in columnas_comunes.items()]
SCHEMA += [(a["target"], TIPO_DUCKDB[a["type"]]) for a in AUDIT_ACTIVE]
SCHEMA_COLS = [c for c, _ in SCHEMA]
assert len(SCHEMA_COLS) == len(set(SCHEMA_COLS)), "Nombres de columna destino duplicados"

# La columna de partición vive en la ruta (fecha_foto=...), no dentro del archivo Parquet
assert COLUMNA_PARTICION in SCHEMA_COLS, f"{COLUMNA_PARTICION} debe estar activa en audit_columns del mapping"
assert COLUMNA_PARTICION == "fecha_foto", "El nombre de la partición se construye a partir de fecha_foto"
SCHEMA_ARCHIVO = [(c, t) for c, t in SCHEMA if c != COLUMNA_PARTICION]
COLS_ARCHIVO = [c for c, _ in SCHEMA_ARCHIVO]
sin_orden = [c for c in ORDENAR_POR if c not in COLS_ARCHIVO]
assert not sin_orden, f"ORDENAR_POR contiene columnas inexistentes en el archivo: {sin_orden}"

log.info("Esquema destino: %d columnas (%d mapeadas + %d auditoría activa %s) | auditoría ignorada: %s",
         len(SCHEMA), len(columnas_comunes), len(AUDIT_ACTIVE), [a["target"] for a in AUDIT_ACTIVE], AUDIT_IGNORED)
log.info("Esquema del archivo Parquet: %d columnas (sin la columna de partición '%s')", len(SCHEMA_ARCHIVO), COLUMNA_PARTICION)
pd.DataFrame(SCHEMA, columns=["columna", "tipo_duckdb"]).groupby("tipo_duckdb").size().to_frame("columnas")

2026-11-17 10:16:56 [INFO] Esquema destino: 114 columnas (113 mapeadas + 1 auditoría activa ['fecha_foto']) | auditoría ignorada: ['nombre_archivo', 'fecha_creacion', 'usuario_creacion']
2026-11-17 10:16:56 [INFO] Esquema del archivo Parquet: 113 columnas (sin la columna de partición 'fecha_foto')


,columnas
tipo_duckdb,
BOOLEAN,8
DATE,19
DOUBLE,21
INTEGER,18
VARCHAR,48


## 4. Transformaciones (`TRF_*`)

In [4]:
def _valores_booleanos(valores) -> set:
    """PyYAML (YAML 1.1) convierte yes/no/true/false sin comillas en bool: se recuperan sus textos originales."""
    out = set()
    for v in valores:
        if v is True:
            out |= {"true", "yes"}
        elif v is False:
            out |= {"false", "no"}
        else:
            out.add(str(v).strip().lower())
    return out


BOOL_TRUE = _valores_booleanos(MAPPING_ANTERIOR["conventions"]["boolean_true_values"])
BOOL_FALSE = _valores_booleanos(MAPPING_ANTERIOR["conventions"]["boolean_false_values"])
assert {"si", "sí"} <= BOOL_TRUE and "no" in BOOL_FALSE, f"Valores booleanos mal interpretados: {BOOL_TRUE} / {BOOL_FALSE}"
log.info("Valores booleanos: verdadero=%s | falso=%s", sorted(BOOL_TRUE), sorted(BOOL_FALSE))
INT32_MIN, INT32_MAX = -(2**31), 2**31 - 1


def _a_texto(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    return s.mask(s.isin(["", "nan", "NaN", "None", "NaT"]))


def trf_none(s):
    return _a_texto(s)


def trf_force_string(s):
    return _a_texto(s)


def trf_amount_space_to_decimal(s):
    """'S/1 002 000 00' -> 1002000.00 ; '1205065 23' -> 1205065.23 (último espacio = decimal)."""
    t = _a_texto(s)
    t = t.str.replace(r"(?i)^s/\s*", "", regex=True)
    t = t.str.replace(r"\s+(\d+)$", r".\1", regex=True)
    t = t.str.replace(r"[\s,]", "", regex=True)
    return pd.to_numeric(t, errors="coerce").astype("Float64")


def trf_date_dayfirst(s):
    t = _a_texto(s)
    return pd.to_datetime(t, errors="coerce", dayfirst=True, format="mixed")


def trf_boolean_si_no(s, valores_true=BOOL_TRUE, valores_false=BOOL_FALSE):
    t = _a_texto(s).str.lower()
    out = pd.Series(pd.NA, index=s.index, dtype="boolean")
    out[t.isin(valores_true).fillna(False)] = True
    out[t.isin(valores_false).fillna(False)] = False
    return out


def trf_integer_strict(s):
    t = _a_texto(s).str.replace(r"[\s,]", "", regex=True)
    n = pd.to_numeric(t, errors="coerce")
    n = n.where((n % 1 == 0) & n.between(INT32_MIN, INT32_MAX))
    return n.astype("Int64")


def trf_id_integer_to_string(s):
    """Evita .0 sin convertir a float ni perder ceros iniciales de identificadores de texto."""
    t = _a_texto(s)
    fraccionario = t.str.fullmatch(r"[+-]?\d+\.\d+", na=False) & ~t.str.fullmatch(r"[+-]?\d+\.0+", na=False)
    return t.str.replace(r"^([+-]?\d+)\.0+$", r"\1", regex=True).mask(fraccionario)


TRANSFORMS = {
    "TRF_ID_INTEGER_TO_STRING": trf_id_integer_to_string,
    "TRF_NONE": trf_none,
    "TRF_FORCE_STRING": trf_force_string,
    "TRF_AMOUNT_SPACE_TO_DECIMAL": trf_amount_space_to_decimal,
    "TRF_DATE_DAYFIRST": trf_date_dayfirst,
    "TRF_BOOLEAN_SI_NO": trf_boolean_si_no,
    "TRF_INTEGER_STRICT": trf_integer_strict,
}
for mapping_id, tabla in MAPAS_COLUMNAS.items():
    faltantes = set(tabla["transform"]) - set(TRANSFORMS)
    assert not faltantes, f"Transformaciones sin implementar ({mapping_id}): {faltantes}"
    log.info("Transformaciones %s: %s", mapping_id, tabla["transform"].value_counts().to_dict())


2026-11-17 10:17:01 [INFO] Valores booleanos: verdadero=['1', 's', 'si', 'sí', 't', 'true', 'y', 'yes'] | falso=['0', 'f', 'false', 'n', 'no']
2026-11-17 10:17:01 [INFO] Transformaciones map_cgr_infobras_obras_publicas: {'TRF_NONE': 41, 'TRF_AMOUNT_SPACE_TO_DECIMAL': 21, 'TRF_INTEGER_STRICT': 18, 'TRF_DATE_DAYFIRST': 18, 'TRF_BOOLEAN_SI_NO': 8, 'TRF_FORCE_STRING': 7}
2026-11-17 10:17:01 [INFO] Transformaciones map_cgr_infobras_obras_publicas_desde_20260701: {'TRF_NONE': 30, 'TRF_AMOUNT_SPACE_TO_DECIMAL': 19, 'TRF_INTEGER_STRICT': 16, 'TRF_DATE_DAYFIRST': 15, 'TRF_BOOLEAN_SI_NO': 8, 'TRF_ID_INTEGER_TO_STRING': 5, 'TRF_FORCE_STRING': 2}


## 5. Lectura de los archivos de origen

- `listar_archivos`: ubica los Excel en `data/raw/cgr` y extrae la **fecha del snapshot desde el nombre** (`DD-MM-YYYY`); los nombres inválidos se omiten y se registran.
- `leer_cabecera`: detecta la fila de cabecera y lee los metadatos previos (incluye la *Fecha de consulta* declarada dentro del Excel, que se contrasta con la fecha del nombre).
- `cargar_archivo`: selecciona la versión por `fecha_foto`, completa los destinos sin fuente con `NULL` y lee el Excel como texto, resuelve encabezados duplicados por posición, aplica las transformaciones y **cuenta los valores que no pudieron convertirse** (origen no vacío → destino nulo) por columna.

In [5]:
PATRON_ARCHIVO = re.compile(MAPPING_ANTERIOR["source"]["snapshot_date_extraction"]["pattern"])


def listar_archivos(fecha_inicio: str | None, fecha_fin: str | None, raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    f_ini = datetime.strptime(str(fecha_inicio or "19000101"), "%Y%m%d").date()
    f_fin = datetime.strptime(str(fecha_fin or "29991231"), "%Y%m%d").date()
    if f_ini > f_fin:
        raise ValueError(f"fecha_inicio ({fecha_inicio}) no puede ser mayor que fecha_fin ({fecha_fin}).")
    registros = []
    for ruta in raw_dir.rglob("DataSet-Obras-Publicas*.xlsx"):
        m = PATRON_ARCHIVO.fullmatch(ruta.name)
        if not m:
            log.warning("Archivo omitido (nombre no coincide con el patrón): %s", ruta.name)
            continue
        dd, mm, yyyy = m.groups()
        try:
            fecha = datetime(int(yyyy), int(mm), int(dd)).date()
        except ValueError:
            log.warning("Archivo omitido (fecha inválida en el nombre): %s", ruta.name)
            continue
        if f_ini <= fecha <= f_fin:
            registros.append({"fecha_foto": fecha, "nombre_archivo": ruta.name, "ruta": ruta})
    df = pd.DataFrame(registros, columns=["fecha_foto", "nombre_archivo", "ruta"])
    return df.sort_values("fecha_foto").reset_index(drop=True)


def sha256_archivo(ruta: Path, bloque: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        while chunk := f.read(bloque):
            h.update(chunk)
    return h.hexdigest()


def leer_cabecera(ruta: Path, fila_cabecera=None, max_filas: int = 15):
    """Retorna (fila_cabecera 1-based, metadatos previos, fecha de consulta declarada en el Excel)."""
    muestra = pd.read_excel(ruta, header=None, nrows=max_filas, dtype=str)
    fila = fila_cabecera
    if fila is None:
        for i in range(len(muestra)):
            valores = {norm_header(v) for v in muestra.iloc[i].dropna()}
            if {"código infobras", "codigo infobras"} & valores:
                fila = i + 1
                break
    if fila is None:
        raise ValueError(f"No se encontró la fila de cabecera en {ruta.name}")
    meta = [" | ".join(str(v) for v in muestra.iloc[i].dropna()) for i in range(fila - 1)]
    fecha_consulta = None
    for linea in meta:
        m = re.search(r"fecha de consulta:\s*(\d{2})/(\d{2})/(\d{4})", linea, re.I)
        if m:
            fecha_consulta = date(int(m[3]), int(m[2]), int(m[1]))
    return fila, meta, fecha_consulta


def filas_segun_dimension(ruta: Path, fila_cabecera: int):
    """Filas de datos según la dimensión declarada en la hoja (control independiente de la lectura con pandas)."""
    try:
        wb = openpyxl.load_workbook(ruta, read_only=True)
        max_row = wb.worksheets[0].max_row
        wb.close()
        return None if max_row is None else int(max_row) - fila_cabecera
    except Exception:
        return None


def cargar_archivo(ruta: Path, fecha_foto, fila_cabecera=None):
    """Lee un Excel y aplica el mapping. Retorna (DataFrame destino, estadísticas de lectura)."""
    mapping, mapping_path = seleccionar_mapping(fecha_foto)
    df_map = MAPAS_COLUMNAS[mapping["mapping"]["id"]]
    valores_true = _valores_booleanos(mapping["conventions"]["boolean_true_values"])
    valores_false = _valores_booleanos(mapping["conventions"]["boolean_false_values"])
    fila, meta, fecha_consulta = leer_cabecera(ruta, fila_cabecera)
    crudo = pd.read_excel(ruta, header=None, skiprows=fila - 1, dtype=str)

    encabezados = crudo.iloc[0].tolist()
    datos = crudo.iloc[1:].reset_index(drop=True)
    ocurrencias, claves = {}, []
    for h in encabezados:
        n = norm_header(h) if pd.notna(h) else ""
        ocurrencias[n] = ocurrencias.get(n, 0) + 1
        claves.append((n, ocurrencias[n]))
    posicion = {clave: i for i, clave in enumerate(claves)}

    # Los campos sin destino confirmado se informan como pendientes y se omiten.
    pendientes = []
    for grupo in mapping["column_groups"]:
        for col in grupo["columns"]:
            if col.get("target") is None:
                pendientes.append(col["source"])
    posiciones_usadas = set()
    ausentes = []
    salida, conversion = {}, []
    for r in df_map.itertuples():
        if pd.notna(r.position):
            pos = int(r.position) - 1
            esperado = re.sub(r"\.\d+$", "", r.source_norm)
            actual = norm_header(encabezados[pos]) if 0 <= pos < len(encabezados) else ""
            if re.sub(r"\.\d+$", "", actual) != esperado:
                raise ValueError(f"Estructura incompatible con {mapping_path.name}: posición {pos + 1}, esperado {r.source!r}, recibido {actual!r}")
        else:
            pos = posicion.get((r.source_norm, r.occurrence))
        if pos is not None:
            posiciones_usadas.add(pos)
        if pos is None:
            ausentes.append(f"{r.source} (ocurrencia {r.occurrence})")
            serie = pd.Series([None] * len(datos), dtype="object")
        else:
            serie = datos.iloc[:, pos]
        n_origen = int(_a_texto(serie).notna().sum())
        out = (trf_boolean_si_no(serie, valores_true, valores_false)
               if r.transform == "TRF_BOOLEAN_SI_NO" else TRANSFORMS[r.transform](serie))
        n_destino = int(out.notna().sum())
        salida[r.target] = out
        conversion.append({"columna": r.target, "transform": r.transform, "n_origen": n_origen, "n_destino": n_destino,
                           "n_falla": n_origen - n_destino,
                           "pct_falla": round((n_origen - n_destino) / n_origen * 100, 3) if n_origen else 0.0})

    sin_mapping = [str(h) for i, h in enumerate(encabezados) if i not in posiciones_usadas]
    df = pd.DataFrame(salida)
    filas_leidas = len(df)
    df = df.dropna(how="all").reset_index(drop=True)

    # Mantener el mismo esquema para ambas estructuras; sin fuente en esta versión -> NULL.
    for columna in columnas_comunes:
        if columna not in df:
            df[columna] = None

    contexto = {"fecha_foto": pd.Timestamp(fecha_foto), "nombre_archivo": ruta.name}
    for a in AUDIT_ACTIVE:
        df[a["target"]] = AUDIT_GENERATORS[a["target"]](contexto)

    stats = {
        "mapping_id": mapping["mapping"]["id"], "mapping_version": mapping["mapping"]["version"],
        "mapping_archivo": mapping_path.name, "pendientes": pendientes,
        "destinos_sin_fuente": [c for c in columnas_comunes if c not in salida],
        "fila_cabecera": fila, "metadatos": meta, "fecha_consulta": fecha_consulta,
        "columnas_origen": len(encabezados), "filas_leidas": filas_leidas,
        "filas_vacias_descartadas": filas_leidas - len(df), "filas_cargadas": len(df),
        "filas_dimension_excel": filas_segun_dimension(ruta, fila),
        "sin_mapping": sin_mapping, "ausentes": ausentes,
        "conversion": pd.DataFrame(conversion),
    }
    return df, stats

## 6. Escritura de la partición Parquet (layout Hive)

- `escribir_particion_tmp` escribe el DataFrame en `data/stg/_work/.../_tmp/fecha_foto=YYYY-MM-DD/part-00000.parquet` con el esquema explícito del mapping (`CAST` por columna), **sin** la columna de partición, ordenado por `ORDENAR_POR` y comprimido con ZSTD.
- `promover_particion` mueve la carpeta `fecha_foto=...` validada al dataset reemplazando la anterior (con rollback si el movimiento falla).
- `rechazar_particion` mueve lo rechazado a `_work/.../_error` para inspección.

In [6]:
def nombre_particion(fecha_foto) -> str:
    """Nombre de carpeta Hive: fecha_foto=YYYY-MM-DD (la fecha ISO se infiere como DATE al leer)."""
    return f"{COLUMNA_PARTICION}={fecha_foto:%Y-%m-%d}"


TMP_ROOT = WORK_DIR / "_tmp"
ERROR_ROOT = WORK_DIR / "_error"
OLD_ROOT = WORK_DIR / "_old"


def escribir_particion_tmp(df: pd.DataFrame, fecha_foto):
    """Retorna (carpeta de partición temporal, archivo parquet)."""
    p = nombre_particion(fecha_foto)
    if TMP_ROOT.exists():
        shutil.rmtree(TMP_ROOT)
    tmp = TMP_ROOT / p
    tmp.mkdir(parents=True)
    archivo = tmp / "part-00000.parquet"
    expr = ", ".join(f'CAST("{c}" AS {t}) AS "{c}"' for c, t in SCHEMA_ARCHIVO)
    orden = (" ORDER BY " + ", ".join(f'"{c}"' for c in ORDENAR_POR)) if ORDENAR_POR else ""
    con = duckdb.connect()
    try:
        con.register("df_part", df)
        con.execute(
            f"COPY (SELECT {expr} FROM df_part{orden}) TO '{archivo.as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION {COMPRESION}, COMPRESSION_LEVEL {int(COMPRESION_NIVEL)}, ROW_GROUP_SIZE 100000)"
        ).fetchall()
    finally:
        con.close()
    return tmp, archivo


def promover_particion(tmp: Path, fecha_foto) -> Path:
    """Reemplaza la partición final por la temporal validada (rollback si falla el movimiento)."""
    p = nombre_particion(fecha_foto)
    final, viejo = DEST_DIR / p, OLD_ROOT / p
    if OLD_ROOT.exists():
        shutil.rmtree(OLD_ROOT)
    OLD_ROOT.mkdir(parents=True)
    if final.exists():
        final.rename(viejo)
    try:
        tmp.rename(final)
    except Exception:
        if viejo.exists() and not final.exists():
            viejo.rename(final)
        raise
    shutil.rmtree(OLD_ROOT, ignore_errors=True)
    shutil.rmtree(TMP_ROOT, ignore_errors=True)
    return final


def rechazar_particion(tmp: Path, fecha_foto) -> Path:
    p = nombre_particion(fecha_foto)
    destino = ERROR_ROOT / p
    if destino.exists():
        shutil.rmtree(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp.rename(destino)
    shutil.rmtree(TMP_ROOT, ignore_errors=True)
    return destino

## 7. Validaciones de la partición

Se validan sobre el Parquet **recién escrito** (antes de promoverlo). Niveles:
- **ERROR** (`FALLA` ⇒ la partición se rechaza): archivo, esquema (nombres/orden/tipos, **sin** la columna de partición dentro del archivo), conteo de filas, conciliación de filas, partición Hive única (`fecha_foto` inferida como `DATE` desde la ruta), paridad de no-nulos columna a columna DataFrame↔Parquet, `codigo_infobras` sin nulos.
- **WARN** (`ALERTA` ⇒ se carga pero queda registrado): reglas de calidad `DQ003–DQ009` del mapping, tasa de fallas de conversión, filas vs dimensión del Excel, columnas sin mapping/ausentes, *Fecha de consulta* del Excel vs fecha del nombre.

In [7]:
def escalar(con, sql: str, params=None):
    return con.execute(sql, params or []).fetchall()[0][0]


def _regla_dq(mapping, rule_id: str):
    return next((r for r in mapping["data_quality"]["rules"] if r["id"] == rule_id), {})


def validar_particion(archivo_parquet: Path, fecha_foto, df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """Retorna un DataFrame con el resultado de cada validación."""
    mapping, _ = seleccionar_mapping(fecha_foto)
    checks = []

    def chk(id_, nivel, descripcion, esperado, obtenido, ok):
        estado = "OK" if ok else ("FALLA" if nivel == "ERROR" else "ALERTA")
        checks.append({"id": id_, "nivel": nivel, "validacion": descripcion,
                       "esperado": str(esperado), "obtenido": str(obtenido), "estado": estado})

    existe = archivo_parquet.exists() and archivo_parquet.stat().st_size > 0
    chk("V01", "ERROR", "Archivo Parquet existe y no está vacío", "> 0 bytes",
        f"{archivo_parquet.stat().st_size:,} bytes" if archivo_parquet.exists() else "no existe", existe)
    if not existe:
        return pd.DataFrame(checks)

    con = duckdb.connect()
    try:
        # Vista Hive: la columna fecha_foto se infiere de la ruta de la partición (como al leer el dataset completo)
        glob_hive = (archivo_parquet.parent.parent / f"{COLUMNA_PARTICION}=*" / "*.parquet").as_posix()
        con.execute(f"CREATE VIEW p AS SELECT * FROM read_parquet('{glob_hive}', hive_partitioning = true)").fetchall()

        # --- Esquema del archivo (sin hive): no debe contener la columna de partición ---
        desc = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{archivo_parquet.as_posix()}', hive_partitioning = false)").fetchall()
        cols_real = [d[0] for d in desc]
        tipos_real = {d[0]: d[1] for d in desc}
        chk("V02", "ERROR", f"Columnas y orden = esquema del mapping (sin '{COLUMNA_PARTICION}' en el archivo)",
            f"{len(COLS_ARCHIVO)} columnas", f"{len(cols_real)} columnas", cols_real == COLS_ARCHIVO)
        dif_tipos = [f"{c}: {tipos_real.get(c)}≠{t}" for c, t in SCHEMA_ARCHIVO if tipos_real.get(c) != t]
        chk("V03", "ERROR", "Tipos de dato = esquema del mapping", "0 diferencias", f"{len(dif_tipos)} diferencias {dif_tipos[:5]}",
            not dif_tipos)

        # --- Filas ---
        n_pq = escalar(con, "SELECT count(*) FROM p")
        chk("V04", "ERROR", "Filas del Parquet = filas cargadas en memoria", len(df), n_pq, n_pq == len(df))
        chk("V05", "ERROR", "Conciliación: leídas = cargadas + descartadas (vacías)", stats["filas_leidas"],
            f"{stats['filas_cargadas']} + {stats['filas_vacias_descartadas']}",
            stats["filas_leidas"] == stats["filas_cargadas"] + stats["filas_vacias_descartadas"])
        chk("V06", "ERROR", "Filas cargadas > 0", "> 0", n_pq, n_pq > 0)

        # --- Partición única ---
        fechas = [str(r[0]) for r in con.execute("SELECT DISTINCT fecha_foto FROM p").fetchall()]
        tipo_part = dict((d[0], d[1]) for d in con.execute("DESCRIBE p").fetchall()).get("fecha_foto")
        chk("V07", "ERROR", "fecha_foto (desde la ruta Hive) única, igual a la partición y de tipo DATE",
            f"{fecha_foto} / DATE", f"{fechas} / {tipo_part}", fechas == [str(fecha_foto)] and tipo_part == "DATE")

        # --- Paridad de no-nulos DataFrame <-> Parquet ---
        agg = ", ".join(f'count("{c}")' for c in SCHEMA_COLS)
        cnt_pq = dict(zip(SCHEMA_COLS, con.execute(f"SELECT {agg} FROM p").fetchall()[0]))
        cnt_df = df[SCHEMA_COLS].notna().sum().to_dict()
        dif = [f"{c}: df={cnt_df[c]} pq={cnt_pq[c]}" for c in SCHEMA_COLS if int(cnt_df[c]) != int(cnt_pq[c])]
        chk("V08", "ERROR", "Paridad de no-nulos por columna (DataFrame vs Parquet)", "0 columnas con diferencia",
            f"{len(dif)} columnas {dif[:5]}", not dif)

        # --- DQ001 ---
        n_null = escalar(con, "SELECT count(*) FROM p WHERE codigo_infobras IS NULL")
        chk("V09", "ERROR", "DQ001 codigo_infobras sin nulos", 0, n_null, n_null == 0)

        # --- Controles de origen (WARN) ---
        dim = stats["filas_dimension_excel"]
        chk("W01", "WARN", "Filas leídas = filas según dimensión de la hoja Excel", dim if dim is not None else "n/d",
            stats["filas_leidas"], dim is None or dim == stats["filas_leidas"])
        fc = stats["fecha_consulta"]
        chk("W02", "WARN", "Fecha de consulta del Excel = fecha del nombre del archivo", str(fecha_foto), str(fc),
            fc is not None and fc == fecha_foto)
        chk("W03", "WARN", "Columnas de origen sin mapping (omitidas)", 0, f"{len(stats['sin_mapping'])} {stats['sin_mapping'][:5]}",
            not stats["sin_mapping"])
        chk("W04", "WARN", "Columnas del mapping ausentes en el origen (NULL)", 0, f"{len(stats['ausentes'])} {stats['ausentes'][:5]}",
            not stats["ausentes"])

        # --- Reglas de calidad del mapping (WARN) ---
        nk = mapping["target"]["natural_key_candidate"]
        keys = ", ".join(f'"{k}"' for k in nk)
        n_dup = escalar(con, f"SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT {keys} FROM p)) FROM p")
        chk("W05", "WARN", f"DQ003 clave natural única {nk}", 0, n_dup, n_dup == 0)

        n_mes = escalar(con, "SELECT count(*) FROM p WHERE mes_avance IS NOT NULL AND mes_avance NOT BETWEEN 1 AND 12")
        chk("W06", "WARN", "DQ004 mes_avance entre 1 y 12", 0, n_mes, n_mes == 0)

        fuera = {}
        for c in _regla_dq(mapping, "DQ005").get("target", []):
            n = escalar(con, f'SELECT count(*) FROM p WHERE "{c}" < 0 OR "{c}" > 100')
            if n:
                fuera[c] = n
        chk("W07", "WARN", "DQ005 porcentajes entre 0 y 100", "0 columnas fuera de rango", fuera or "0", not fuera)

        n_f = escalar(con, "SELECT count(*) FROM p WHERE fecha_inicio_obra > fecha_fin_programada_obra")
        chk("W08", "WARN", "DQ006 fecha_inicio_obra <= fecha_fin_programada_obra", 0, n_f, n_f == 0)

        malos = {}
        for c in _regla_dq(mapping, "DQ009").get("target", []):
            n = escalar(con, f'SELECT count(*) FROM p WHERE "{c}" IS NOT NULL AND NOT regexp_full_match("{c}", ?)', ["[0-9]{11}"])
            if n:
                malos[c] = n
        chk("W09", "WARN", "DQ009 RUC de 11 dígitos", "0 registros inválidos", malos or "0", not malos)

        umbral = _regla_dq(mapping, "DQ008").get("threshold_pct", 1)
        conv = stats["conversion"]
        sobre = conv[conv["pct_falla"] > umbral]
        detalle = {r.columna: f"{r.pct_falla}% ({r.n_falla})" for r in sobre.itertuples()}
        chk("W10", "WARN", f"DQ008 fallas de conversión <= {umbral}% por columna", "0 columnas sobre el umbral",
            f"{len(detalle)} {dict(list(detalle.items())[:5])}", sobre.empty)
    finally:
        con.close()
    return pd.DataFrame(checks)

## 8. Proceso por snapshot y ejecución

`procesar_snapshot` ejecuta, para cada archivo: **leer → transformar → escribir temporal → validar → promover (o rechazar)** y registra cada paso en el log.

Estados: `CARGADO` · `CARGADO_CON_ALERTAS` · `OMITIDO` (partición existente y `SOBRESCRIBIR = False`) · `RECHAZADO` (alguna validación de nivel ERROR falló) · `ERROR` (excepción).

In [8]:
RESULTADOS = {}   # detalle por fecha_foto (validaciones, conversión, estadísticas) para inspección posterior


def procesar_snapshot(fila) -> dict:
    t0 = time.time()
    fecha, ruta, nombre = fila.fecha_foto, Path(fila.ruta), fila.nombre_archivo
    mapping, mapping_path = seleccionar_mapping(fecha)
    p = nombre_particion(fecha)
    res = {"fecha_foto": fecha, "particion": p, "archivo": nombre, "estado": None, "filas_leidas": None,
           "filas_cargadas": None, "validaciones_falla": None, "validaciones_alerta": None,
           "tamano_parquet_mb": None, "segundos": None, "sha256_origen": None, "detalle": ""}
    res.update(mapping_id=mapping["mapping"]["id"], mapping_version=mapping["mapping"]["version"], mapping_archivo=mapping_path.name)
    log.info("-" * 100)
    log.info("[%s] Mapping: %s v%s | %s", p, res["mapping_id"], res["mapping_version"], res["mapping_archivo"])
    log.info("[%s] Archivo: %s", p, ruta)
    try:
        res["sha256_origen"] = sha256_archivo(ruta)
        log.info("[%s] Origen: %.1f MB | sha256=%s", p, ruta.stat().st_size / 1024**2, res["sha256_origen"])

        if (DEST_DIR / p).exists() and not SOBRESCRIBIR:
            res.update(estado="OMITIDO", detalle="La partición ya existe y SOBRESCRIBIR=False")
            log.warning("[%s] OMITIDO: la partición ya existe y SOBRESCRIBIR=False", p)
            return res
        if (DEST_DIR / p).exists():
            log.info("[%s] La partición existe y será reemplazada (snapshot_replace)", p)

        t = time.time()
        df, stats = cargar_archivo(ruta, fecha, FILA_CABECERA)
        log.info("[%s] Lectura+transformación: %.1f s | cabecera en fila %s | fecha de consulta en Excel=%s | columnas origen=%s",
                 p, time.time() - t, stats["fila_cabecera"], stats["fecha_consulta"], stats["columnas_origen"])
        log.info("[%s] Filas leídas=%s | descartadas (vacías)=%s | cargadas=%s | dimensión Excel=%s",
                 p, f"{stats['filas_leidas']:,}", stats["filas_vacias_descartadas"], f"{stats['filas_cargadas']:,}",
                 stats["filas_dimension_excel"])
        if stats["sin_mapping"]:
            log.warning("[%s] Columnas de origen sin mapping (omitidas): %s", p, stats["sin_mapping"])
        if stats["ausentes"]:
            log.warning("[%s] Columnas del mapping ausentes en origen (NULL): %s", p, stats["ausentes"])
        if stats["destinos_sin_fuente"]:
            log.info("[%s] Destinos sin fuente en esta versión (NULL): %s", p, stats["destinos_sin_fuente"])
        if stats["pendientes"]:
            log.warning("[%s] Campos sin destino confirmado (omitidos): %s", p, stats["pendientes"])
        conv = stats["conversion"]
        for r in conv[conv["n_falla"] > 0].sort_values("n_falla", ascending=False).itertuples():
            log.warning("[%s] Conversión %s (%s): %s de %s valores no convertidos (%s%%)",
                        p, r.columna, r.transform, f"{r.n_falla:,}", f"{r.n_origen:,}", r.pct_falla)

        t = time.time()
        tmp, archivo = escribir_particion_tmp(df, fecha)
        log.info("[%s] Parquet temporal escrito: %s (%.1f MB, %.1f s)", p, archivo, archivo.stat().st_size / 1024**2, time.time() - t)

        val = validar_particion(archivo, fecha, df, stats)
        for r in val.itertuples():
            msg = "[%s] %s %s %s | esperado=%s | obtenido=%s -> %s"
            args = (p, r.id, r.nivel, r.validacion, r.esperado, r.obtenido, r.estado)
            (log.info if r.estado == "OK" else log.error if r.estado == "FALLA" else log.warning)(msg, *args)
        n_falla = int((val["estado"] == "FALLA").sum())
        n_alerta = int((val["estado"] == "ALERTA").sum())
        RESULTADOS[fecha] = {"validacion": val, "conversion": conv, "stats": stats, "resumen": res}
        res.update(filas_leidas=stats["filas_leidas"], filas_cargadas=stats["filas_cargadas"],
                   validaciones_falla=n_falla, validaciones_alerta=n_alerta)

        if n_falla:
            rechazado = rechazar_particion(tmp, fecha)
            res.update(estado="RECHAZADO", detalle=f"{n_falla} validación(es) en FALLA; rechazado en {rechazado}")
            log.error("[%s] RECHAZADO: %d validación(es) de nivel ERROR fallaron. Se conserva la partición anterior. Rechazado en: %s",
                      p, n_falla, rechazado)
        else:
            final = promover_particion(tmp, fecha)
            res["tamano_parquet_mb"] = round(sum(f.stat().st_size for f in final.glob("*.parquet")) / 1024**2, 2)
            res["estado"] = "CARGADO_CON_ALERTAS" if n_alerta else "CARGADO"
            log.info("[%s] %s: %s (%s filas, %.1f MB, %d alerta(s))", p, res["estado"], final,
                     f"{stats['filas_cargadas']:,}", res["tamano_parquet_mb"], n_alerta)
        del df
    except Exception as e:
        res.update(estado="ERROR", detalle=f"{type(e).__name__}: {e}")
        log.error("[%s] ERROR: %s: %s\n%s", p, type(e).__name__, e, traceback.format_exc())
        shutil.rmtree(TMP_ROOT, ignore_errors=True)
    finally:
        res["segundos"] = round(time.time() - t0, 1)
        log.info("[%s] Fin (%s) en %.1f s", p, res["estado"], res["segundos"])
        gc.collect()
    return res


def ejecutar(archivos: pd.DataFrame) -> pd.DataFrame:
    resumen = []
    for fila in archivos.itertuples():
        r = procesar_snapshot(fila)
        resumen.append(r)
        if r["estado"] in ("ERROR", "RECHAZADO") and DETENER_ANTE_ERROR:
            log.error("Corrida detenida por DETENER_ANTE_ERROR=True (archivo %s, estado %s)", r["archivo"], r["estado"])
            break
    return pd.DataFrame(resumen)

## 9. Ejecución

In [9]:
if MODO == "PRUEBA":
    archivos = listar_archivos(FECHA_PRUEBA, FECHA_PRUEBA)
    assert len(archivos) == 1, f"MODO=PRUEBA requiere exactamente 1 archivo para {FECHA_PRUEBA}; encontrados: {len(archivos)}"
else:
    archivos = listar_archivos(FECHA_INICIO, FECHA_FIN)
    assert len(archivos) > 0, f"No hay archivos en {RAW_DIR} para el rango {FECHA_INICIO}-{FECHA_FIN}"

log.info("Archivos a procesar: %d | %s", len(archivos), [nombre_particion(f) for f in archivos["fecha_foto"]])
t_total = time.time()
resumen_carga = ejecutar(archivos)
log.info("=" * 100)
log.info("FIN DE CORRIDA | run_id=%s | modo=%s | duración=%.1f s | estados=%s",
         RUN_ID, MODO, time.time() - t_total, resumen_carga["estado"].value_counts().to_dict())
resumen_carga.to_csv(RESUMEN_PATH, index=False, encoding="utf-8-sig")
log.info("Resumen guardado: %s", RESUMEN_PATH)
resumen_carga.drop(columns=["sha256_origen"])

2026-11-17 10:17:33 [INFO] Archivos a procesar: 4 | ['fecha_foto=2026-08-01', 'fecha_foto=2026-09-01', 'fecha_foto=2026-10-01', 'fecha_foto=2026-11-01']
2026-11-17 10:17:33 [INFO] ----------------------------------------------------------------------------------------------------
2026-11-17 10:17:33 [INFO] [fecha_foto=2026-08-01] Mapping: map_cgr_infobras_obras_publicas_desde_20260701 v2.0.0 | stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml
2026-11-17 10:17:33 [INFO] [fecha_foto=2026-08-01] Archivo: C:\ws\formacion\uni-mia\mia-tesis-opa\data\raw\cgr\DataSet-Obras-Publicas 01-08-2026.xlsx
2026-11-17 10:17:33 [INFO] [fecha_foto=2026-08-01] Origen: 54.7 MB | sha256=93d0e71473081a6b249115350520b34c8376a61fe9cb6352ea08a5e021b1d011
2026-11-17 10:17:33 [INFO] [fecha_foto=2026-08-01] La partición existe y será reemplazada (snapshot_replace)
2026-11-17 10:21:05 [INFO] [fecha_foto=2026-08-01] Lectura+transformación: 212.4 s | cabecera en fila 4 | fecha de consult

,fecha_foto,particion,archivo,estado,filas_leidas,filas_cargadas,validaciones_falla,validaciones_alerta,tamano_parquet_mb,segundos,detalle,mapping_id,mapping_version,mapping_archivo
0,2026-08-01,fecha_foto=2026-08-01,DataSet-Obras-Publicas 01-08-2026.xlsx,CARGADO_CON_ALERTAS,191180,191180,0,3,29.7,217.2,,map_cgr_infobras_obras_publicas_desde_20260701,2.0.0,stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml
1,2026-09-01,fecha_foto=2026-09-01,DataSet-Obras-Publicas 01-09-2026.xlsx,CARGADO_CON_ALERTAS,191180,191180,0,3,29.7,224.7,,map_cgr_infobras_obras_publicas_desde_20260701,2.0.0,stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml
2,2026-10-01,fecha_foto=2026-10-01,DataSet-Obras-Publicas 01-10-2026.xlsx,CARGADO_CON_ALERTAS,191180,191180,0,3,29.7,244.2,,map_cgr_infobras_obras_publicas_desde_20260701,2.0.0,stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml
3,2026-11-01,fecha_foto=2026-11-01,DataSet-Obras-Publicas 01-11-2026.xlsx,CARGADO_CON_ALERTAS,191180,191180,0,3,29.7,249.3,,map_cgr_infobras_obras_publicas_desde_20260701,2.0.0,stg__dataset_obras_publicas__desde_20260701__to__sa_obras_publicas_infobras.yaml


## 10. Validación detallada (archivo de prueba)

Resultado de cada validación del último archivo procesado, fallas de conversión por columna y una muestra leída **desde el Parquet** de la partición.

In [10]:
if RESULTADOS:
    ultima_fecha = max(RESULTADOS)
    det = RESULTADOS[ultima_fecha]
    print(f"Partición validada: {nombre_particion(ultima_fecha)} | estado: {det['resumen']['estado']}")
    display(det["validacion"])
else:
    print("Sin resultados de validación (¿archivo omitido o con error?). Revise el log.")

Partición validada: fecha_foto=2026-11-01 | estado: CARGADO_CON_ALERTAS


,id,nivel,validacion,esperado,obtenido,estado
0,V01,ERROR,Archivo Parquet existe y no está vacío,> 0 bytes,"31,142,821 bytes",OK
1,V02,ERROR,Columnas y orden = esquema del mapping (sin 'fecha_foto' en el archivo),113 columnas,113 columnas,OK
2,V03,ERROR,Tipos de dato = esquema del mapping,0 diferencias,0 diferencias [],OK
3,V04,ERROR,Filas del Parquet = filas cargadas en memoria,191180,191180,OK
4,V05,ERROR,Conciliación: leídas = cargadas + descartadas (vacías),191180,191180 + 0,OK
5,V06,ERROR,Filas cargadas > 0,> 0,191180,OK
6,V07,ERROR,"fecha_foto (desde la ruta Hive) única, igual a la partición y de tipo DATE",2026-11-01 / DATE,['2026-11-01'] / DATE,OK
7,V08,ERROR,Paridad de no-nulos por columna (DataFrame vs Parquet),0 columnas con diferencia,0 columnas [],OK
8,V09,ERROR,DQ001 codigo_infobras sin nulos,0,0,OK
9,W01,WARN,Filas leídas = filas según dimensión de la hoja Excel,n/d,191180,OK


In [11]:
if RESULTADOS:
    conv = RESULTADOS[max(RESULTADOS)]["conversion"]
    print("Columnas con valores no convertidos (origen no vacío -> destino nulo):")
    display(conv[conv["n_falla"] > 0].sort_values("n_falla", ascending=False))

Columnas con valores no convertidos (origen no vacío -> destino nulo):


,columna,transform,n_origen,n_destino,n_falla,pct_falla


In [12]:
# Muestra leída desde el Parquet promovido
if RESULTADOS and (DEST_DIR / nombre_particion(max(RESULTADOS))).exists():
    f_pq = (DEST_DIR / nombre_particion(max(RESULTADOS)) / "*.parquet").as_posix()
    con = duckdb.connect()
    print("Esquema leído SIN hive (archivo puro, sin fecha_foto):")
    display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{f_pq}', hive_partitioning = false)").df().head(5))
    print("Muestra leída CON hive (fecha_foto reconstruida desde la ruta):")
    display(con.execute(f"SELECT fecha_foto, * EXCLUDE (fecha_foto) FROM read_parquet('{f_pq}', hive_partitioning = true) LIMIT 5").df())
    con.close()

Esquema leído SIN hive (archivo puro, sin fecha_foto):


,column_name,column_type,null,key,default,extra
0,codigo_entidad,VARCHAR,YES,None,None,None
1,entidad_publica,VARCHAR,YES,None,None,None
2,codigo_infobras,VARCHAR,YES,None,None,None
3,nombre_obra,VARCHAR,YES,None,None,None
4,modalidad_ejecucion_obra,VARCHAR,YES,None,None,None


Muestra leída CON hive (fecha_foto reconstruida desde la ruta):


,fecha_foto,codigo_entidad,entidad_publica,codigo_infobras,nombre_obra,modalidad_ejecucion_obra,flg_corresponde_saldo_obra,codigo_infobras_obra_inicial,naturaleza_obra,tipo_obra_nivel_1,tipo_obra_nivel_2,tipo_obra_nivel_3,estado_ejecucion,flg_reconstruccion_con_cambios,flg_reactivacion_economica,otra_marca,n_informes_monitores_ciudadanos,n_denuncias_viculadas,n_informes_control,n_comentarios_ciudadanos,flg_obra_caracter_reservado,nivel_gobierno,sector_entidad,codigo_unico,codigo_snip,nombre_proyecto,estado_proyecto,tipo_formato,monto_viable_aprobado,fecha_aprobacion,...,numero_dias_paralizado,tipo_certificado_inversion_publica,numero_cipril_cipgn,fech_cipril_cipgn,monto_cipril_cipgn,n_modificaciones,n_dias_modificaciones_plazo,nuevo_plazo_ejecucion_dias,fecha_finalizacion_reprogramada_obra,n_controversias,n_adicionales_obra,monto_adicionales_obra_soles,n_adicionales_supervision,monto_adicionales_supervision_soles,n_deductivos_obra,monto_deductivo_obra_soles,n_deductivo_supervision,monto_deductivos_supervision_soles,fecha_finalizacion_real,motivo_caso_no_llegue_100_porciento,flg_tiene_recepcion_total,fecha_recepcion,flg_tiene_liquidacion_obra,fecha_aprobacion_liquidacion_obra,costo_obra_soles,flg_corresponde_transferencia_otra_entidad,fecha_transferencia,tipo_entidad_recepciona_obra,ano_primer_devengado,monto_total_devengado_proyecto
0,2026-11-01,0282,MINISTERIO DEL INTERIOR,10,CONSTRUCCIÓN E IMPLEMENTACIÓN DE LA COMISARÍA PNP CHINCHEROS APURÍMAC,Contrata,False,None,Construcción/Creación,Orden Público/Defensa Y Seguridad,Orden Público Y Seguridad,Comisaria,Finalizado,True,False,None,<NA>,<NA>,0,0,False,GOBIERNO NACIONAL,INTERIOR,2111695,29795,CONSTRUCCION E IMPLEMENTACION DE LA COMISARIA PNP CHINCHEROS APURIMAC,APROBADO,NaN,1863468.0,NaT,...,<NA>,None,None,NaT,NaN,0,0,120,2012-07-06,0,0,0.0,0,0.0,2,3649.62,0,0.0,2012-09-17,Otros,True,2012-09-30,True,2013-12-23,1088869.83,<NA>,NaT,None,<NA>,NaN
1,2026-11-01,2161,MUNICIPALIDAD DISTRITAL DE MIRAFLORES LIMA,100,REHABILITACION Y MEJORAMIENTO DE LA INFRAESTRUCTURA VIAL DE LAS ZONAS 3A 5A Y 5B DISTR...,Contrata,False,None,Rehabilitación,Transportes Y Comunicaciones,Transporte Urbano,Pista Y Vereda,Finalizado,False,False,None,<NA>,<NA>,0,0,False,GOBIERNO LOCAL,GOBIERNOS LOCALES,2060550,74575,REHABILITACION Y MEJORAMIENTO DE LA INFRAESTRUCTURA VIAL DE LAS ZONAS 3A 5A 5B DEL DIS...,APROBADO,NaN,2303895.0,NaT,...,<NA>,None,None,NaT,NaN,0,0,0,2012-12-10,0,0,0.0,0,0.0,0,0.00,0,0.0,2012-03-08,Otros,True,2012-03-31,True,2012-08-08,2037505.74,<NA>,NaT,None,<NA>,NaN
2,2026-11-01,5650,GOBIERNO REGIONAL METROPOLITANO DE LIMA,1000,EJECUCIÓN DE LA OBRA REHABILITACIÓN Y MEJORAMIENTO DE LA AV CANTA CALLAO TRAMO AV BERT...,Contrata,False,None,Mejoramiento,Transportes Y Comunicaciones,Transporte Urbano,Calles,Finalizado,False,False,None,<NA>,<NA>,1,0,False,GOBIERNO REGIONAL,GOBIERNOS REGIONALES,2109543,138585,REHABILITACION Y MEJORAMIENTO DE LA AV CANTA CALLAO TRAMO AV BERTELLO AV NARANJAL,APROBADO,NaN,37617018.0,NaT,...,<NA>,None,None,NaT,NaN,0,0,180,2013-09-20,0,0,0.0,0,0.0,0,0.00,0,0.0,2013-06-29,Otros,True,2013-06-29,<NA>,NaT,NaN,<NA>,NaT,None,<NA>,NaN
3,2026-11-01,2409,MUNICIPALIDAD DISTRITAL DE LA ARENA,10000,SUSBSISTEMA DE DISTRIBUCIÓN PRIMARIA 10KV 10 SUBSISTEMA DE DISTRIBUCIÓN SECUNDARIA 440...,Administración directa,False,None,Mejoramiento,Energía Y Minas,Energía Eléctrica,Sistema De Distribución Eléctrica,Finalizado,False,False,None,<NA>,<NA>,0,0,False,GOBIERNO LOCAL,GOBIERNOS LOCALES,0,0,NaN,NaN,NaN,NaN,NaT,...,<NA>,None,None,NaT,NaN,0,0,0,2013-03-01,0,0,0.0,0,0.0,0,0.00,0,0.0,2021-12-06,Otros,True,2021-12-06,True,2021-12-06,49433.99,<NA>,NaT,None,<NA>,NaN
4,2026-11-01,2258,MUNICIPALIDAD DISTRITAL DE SAN BARTOLOME,100005,CREACION DE MURO DE CONTENCION PARA ESTABILIZACION DE TALUD INESTABLE DEL PASAJE GRANA...,Administración directa,False,None,Construcción/Creación,Vivienda Construcción Y Saneamiento,Desarrollo Urbano Y Rural,Otra Infraestructura,Finalizado,False,False,None,<NA>,<NA>,0,0,Fal

## 11. Particiones existentes en staging

Lectura del dataset **completo** con DuckDB y `hive_partitioning = true`; el filtro por `fecha_foto` poda carpetas (solo abre las particiones del rango).

In [13]:
archivos_pq = sorted(DEST_DIR.glob(f"{COLUMNA_PARTICION}=*/*.parquet"))
if archivos_pq:
    glob_pq = (DEST_DIR / "*" / "*.parquet").as_posix()
    con = duckdb.connect()
    display(con.execute(
        f"SELECT fecha_foto, count(*) AS registros, count(DISTINCT codigo_infobras) AS obras "
        f"FROM read_parquet('{glob_pq}', hive_partitioning = true) GROUP BY ALL ORDER BY 1"
    ).df())
    # Ejemplo de filtro limpio por partición (solo se leen las carpetas del rango)
    # SELECT ... FROM read_parquet(glob, hive_partitioning = true) WHERE fecha_foto BETWEEN '2025-06-01' AND '2025-12-01'
    con.close()
else:
    print("Aún no hay particiones en", DEST_DIR)

,fecha_foto,registros,obras
0,2023-06-29,153471,153272
1,2023-06-30,153480,153281
2,2023-07-01,153497,153298
3,2023-08-01,154761,154549
4,2024-01-01,160231,160084
5,2024-02-01,160932,160785
6,2024-03-01,161586,161586
7,2024-04-01,162844,162843
8,2024-05-01,163846,163841
9,2024-06-01,165176,165171


## 12. Log de la corrida

In [14]:
print(f"Log: {LOG_PATH}\n")
print("\n".join(LOG_PATH.read_text(encoding="utf-8").splitlines()[-60:]))

Log: C:\ws\formacion\uni-mia\mia-tesis-opa\data\stg\logs\carga_stg_sa_dataset_obras_publicas_completa_20261117_101645.log

2026-11-17 10:28:54 [WARNING] [fecha_foto=2026-10-01] Columnas de origen sin mapping (omitidas): ['N° de Veedurías Ciudadanas', 'Costo Actualizado de la inversión']
2026-11-17 10:28:54 [INFO] [fecha_foto=2026-10-01] Destinos sin fuente en esta versión (NULL): ['n_informes_monitores_ciudadanos', 'n_denuncias_viculadas', 'fecha_aprobacion', 'monto_viable_actualizado', 'fecha_actualizacion', 'tipo_documento_identidad', 'numero_documento_identidad', 'colegiatura', 'numero_colegiatura', 'tipo_documento_identidad_2', 'numero_documento_identidad_2', 'colegiatura_2', 'numero_colegiatura_2', 'estado_actualizacion_registro_avance', 'tipo_certificado_inversion_publica', 'numero_cipril_cipgn', 'fech_cipril_cipgn', 'monto_cipril_cipgn']
2026-11-17 10:28:54 [WARNING] [fecha_foto=2026-10-01] Campos sin destino confirmado (omitidos): ['N° de Veedurías Ciudadanas', 'Costo Actualiza